### **Chatbot with message summarization**

In [ ]:
%%capture --no-stderr
%pip install --quiet -U langchain_core langgraph langchain_openai

In [67]:
import os, openai
from dotenv import load_dotenv, find_dotenv
from langchain_openai import ChatOpenAI

_ = load_dotenv(find_dotenv())
openai.api_key = os.getenv('OPENAI_API_KEY')

llm_model = 'gpt-4o'
chat = ChatOpenAI(model = llm_model)

In [88]:
from langgraph.graph import MessagesState
class State(MessagesState):
    summary: str

In [127]:
from langchain_core.messages import SystemMessage, HumanMessage, RemoveMessage
from typing import Literal
from langgraph.graph import END

# Define the logic to call the model
def call_model(state: State):
    # Get summary if it exists
    summary = state.get('summary', '')

    # If there is summary, then we add it
    if summary:
        
        # Add summary to system message
        system_message = f'Summary of conversation earlier: {summary}'

        # Append summary to any newer messages
        messages = [SystemMessage(content = system_message)] + state['messages']
    
    else:
        messages = state['messages']
    
    response = chat.invoke(messages)
    return {'messages': response}

def summarize_conversation(state: State):
    # First, we get any existing summary
    summary = state.get('summary', '')

    # Create our summarization prompt 
    if summary:
        
        # A summary already exists
        summary_message = (
            f'This is summary of the conversation to date: {summary}\n\n'
            'Extend the summary by taking into account the new messages above:'
        )
        
    else:
        summary_message = 'Create a summary of the conversation above:'

    # Add prompt to our history
    messages = state['messages'] + [HumanMessage(content = summary_message)]
    response = chat.invoke(messages)
    
    # Delete all but the 2 most recent messages
    delete_messages = [RemoveMessage(id = m.id) for m in state['messages'][:-2]]
    return {'summary': response.content, 'messages': delete_messages}

import random
# Determine whether to end or summarize the conversation
def should_continue(state: State) -> Literal['summarize_conversation', END]:
    '''Return the next node to execute.'''
    
    messages = state['messages']
    
    # If there are more than six messages, then we summarize the conversation
    if len(messages) > 6:
        return 'summarize_conversation'
    
    # Otherwise we can just end
    return END

#### **Adding memory**
State in LangGraph is transient and limited to a single execution, which restricts support for multi-turn conversations with interruptions. To overcome this, LangGraph offers built-in persistence via a checkpointer, enabling automatic state saving after each step. This allows the graph to maintain memory and resume from the latest state.

In [ ]:
from IPython.display import Image, display
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import StateGraph, START

# Define a new graph
builder = StateGraph(State)
builder.add_node('conversation', call_model)
builder.add_node('summarize_conversation', summarize_conversation)

# Set the entrypoint as conversation
builder.add_edge(START, 'conversation')
builder.add_conditional_edges('conversation', should_continue)
builder.add_edge('summarize_conversation', END)

# Compile
memory = MemorySaver()
graph = builder.compile(checkpointer = memory)
display(Image(graph.get_graph().draw_mermaid_png()))

#### **Threads**

In [ ]:
# Create a thread
config = {'configurable': {'thread_id': '1'}}

# Start conversation
input_message = HumanMessage(content='hi! I\'m Lance')
output = graph.invoke({'messages': [input_message]}, config)

input_message = HumanMessage(content='what\'s my name?')
output = graph.invoke({'messages': [input_message]}, config)

input_message = HumanMessage(content='i like the 49ers!')
output = graph.invoke({'messages': [input_message]}, config)

In [ ]:
graph.get_state(config).values.get('summary','')

In [ ]:
input_message = HumanMessage(content='''i like Nick Bosa, isn't he the highest paid defensive player?''')
output = graph.invoke({'messages': [input_message]}, config) 


In [ ]:
graph.get_state(config).values.get('summary','')